# Large Scale Ingestion (SSD Optimized)

This notebook is optimized for processing large datasets (10GB - 1TB) by:
1. **Streaming Data**: Processing files one-by-one to avoid crashing RAM.
2. **Local Embeddings**: Using HuggingFace (free, fast, private) to avoid API costs and rate limits.
3. **SSD Storage**: utilizing the large storage on your E: drive.

### Prerequisites
1. Ensure you have copied your data to `E:\WMS_selection` (which appears as `/mnt/e/WMS_selection` here).


In [ ]:
import os
import glob
import time
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Disable parallelism warnings for tokenizers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
# --- CONFIGURATION ---
# 1. Input Data Path (On your SSD)
SOURCE_DATA_PATH = "/mnt/e/WMS_selection"

# 2. Database Save Path (On your SSD)
DB_SAVE_PATH = "/mnt/e/chroma_db_wms"

# 3. Local Model Name (Small, Fast, Effective)
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

print(f"Reading data from: {SOURCE_DATA_PATH}")
print(f"Saving Database to: {DB_SAVE_PATH}")

In [ ]:
# Initialize the Local Embedding Model
# This downloads the model once (~100MB) and runs on your machine.
print("Loading embedding model... (this may take a moment the first time)")
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME)

# Initialize Vector Store (Persistent on SSD)
vectorstore = Chroma(
    persist_directory=DB_SAVE_PATH,
    embedding_function=embeddings
)
print("Vector Store Ready.")

In [ ]:
# --- ROBUST INGESTION LOOP ---
# This loop is "Memory Safe". It loads, splits, and embeds ONE file at a time.

# 1. Find all files (Recursive search)
#    NOTE: 105GB might have non-text files. We filter for common text extensions.
#    You can add 'pdf' to this list if you install the 'pypdf' library.
query_path = os.path.join(SOURCE_DATA_PATH, "**")
extensions = {'.md', '.txt', '.csv', '.py', '.json', '.html'}

print("Scanning for files... (this might take a minute on 105GB)")
all_files = glob.glob(query_path, recursive=True)
target_files = [f for f in all_files if os.path.splitext(f)[1].lower() in extensions]

print(f"Found {len(all_files)} total files.")
print(f"Processing {len(target_files)} text/code files.")

# 2. Setup Splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# 3. Process Loop
BATCH_SIZE = 100 # Commit to DB every 100 docs to speed things up
batch_chunks = []
start_time = time.time()

for i, file_path in enumerate(target_files):
    try:
        # Load ONE file
        loader = TextLoader(file_path, encoding='utf-8', autodetect_encoding=True)
        docs = loader.load()
        
        # Split ONE file
        chunks = text_splitter.split_documents(docs)
        
        # Add to batch
        batch_chunks.extend(chunks)
        
        # If batch is big enough, write to DB
        if len(batch_chunks) >= BATCH_SIZE:
            vectorstore.add_documents(batch_chunks)
            batch_chunks = [] # Clear memory
            
        # Progress Update
        if i % 100 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            print(f"Progress: {i}/{len(target_files)} files ({rate:.2f} files/sec)...")
            
    except Exception as e:
        # Don't stop the whole process just because one file is weird
        print(f"Skipping {os.path.basename(file_path)}: {str(e)[:100]}")

# Process any remaining chunks
if batch_chunks:
    vectorstore.add_documents(batch_chunks)

print("DONE! All files processed.")

In [ ]:
# Verify the count
count = vectorstore._collection.count()
print(f"Total vectors in database: {count:,}")

# Perform a Test Search
results = vectorstore.similarity_search("What is the main requirement?", k=2)
for res in results:
    print("\n--- Result ---")
    print(res.page_content[:200])
    print(f"Source: {res.metadata['source']}")